# Notebook 32 — Four-Deck Sequential Random Forest Stacking

**Research-only.** Implements the proposed deck-to-deck learning mechanism without shuffling market dates. Randomly shuffling ticker/date rows would create temporal leakage even if each stage trained on a disjoint subset. We use four chronological global-date decks and score only later development folds. The closed final holdout and prospective ledger remain untouched.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise FileNotFoundError('Run inside stock_ml_dynamic')
sys.path.insert(0,str(ROOT/'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.sequential_deck_stacking import run_deck_research
SEED=42
CAP=60000
TREES=200
LEAF=8
CLIP=(.005,.995)
THRESHOLD=.45
COST_BPS=10
OUTPUT=ROOT/'artifacts'/'notebook32_four_decks'


## 1. Load the same 1-day master dataset as Notebook 31

In [ ]:
paths=[ROOT/'artifacts/master_tables/1d',ROOT/'notebooks/artifacts/master_tables/1d']
source=next((p for p in paths if (p/'master_labeled.parquet').exists()),None)
if source is None: raise FileNotFoundError('Missing 1d master_labeled.parquet')
master,latest,bundle,manifest=load_master(source)
print('Rows:',len(master),'Tickers:',master.symbol.nunique())


## 2. Fit four sequential decks within each chronological outer fold

This trains **18 Random Forest models**: 4 sequential stages plus 2 baseline models for each of 3 outer folds. The final 20% historical holdout remains excluded. Outlier bounds are fitted separately for each stage using only its own deck.

In [ ]:
results=run_deck_research(master,n_outer=3,cap=CAP,trees=TREES,leaf=LEAF,seed=SEED,
                          clip=CLIP,threshold=THRESHOLD,cost_bps=COST_BPS,output_dir=OUTPUT)
print('Closed historical holdout starts after:',results['cutoff'])
print('Base feature count:',results['feature_count'])
display(results['audit'])


## 3. Out-of-sample comparison with conventional Random Forest baselines

In [ ]:
outer=results['outer']
display(outer.round(5))
summary=outer.groupby('model',as_index=False).agg(accuracy=('accuracy','mean'),auc=('auc','mean'),
     logloss=('logloss','mean'),brier_up=('brier_up','mean'),coverage=('coverage','mean'),
     up_precision=('up_precision','mean'),mean_net=('mean_net','mean'),entry_count=('entry_count','sum'))
display(summary.sort_values('accuracy',ascending=False).round(5))


## 4. Visual comparison and fold stability

In [ ]:
for metric in ('accuracy','auc','logloss','up_precision','coverage','mean_net'):
    fig,ax=plt.subplots(figsize=(10,4))
    for model,part in outer.groupby('model'):
        ax.plot(part.fold,part[metric],marker='o',label=model)
    ax.set_title(f'Outer-fold {metric}');ax.set_xlabel('Chronological fold');ax.set_xticks(sorted(outer.fold.unique()))
    ax.legend();ax.grid(alpha=.2);plt.show()


## 5. Audit the deck boundaries and model-feature progression

In [ ]:
audit=results['audit'].copy()
display(audit[['fold','stage','train_rows','train_start','train_label_end','features','up_rate']])
print('Feature counts per stage:',audit.groupby('stage').features.unique().to_dict())


## 6. Research decision

A useful cascade should beat **both** the deck-4-only model and the all-history baseline on later chronological folds, not merely fit the training decks better. Inspect accuracy, AUC, calibration, signal coverage and net returns. All selected trades are correlated by date, so these results are not proof of a tradable edge.

In [ ]:
pivot=outer.pivot(index='fold',columns='model',values='accuracy')
display(pivot)
for other in ('deck4_only_rf','all_training_rf'):
    wins=int((pivot['sequential_four_deck']>pivot[other]).sum())
    print(f'Sequential model beats {other} on {wins}/{len(pivot)} folds')
print('Saved diagnostics:',OUTPUT)
